# BankAssist Triage — Intent Classification Modelling

## Objective

This notebook develops and evaluates the traditional machine learning intent classifier for the BankAssist Triage application.

The modelling stage uses the fixed train, validation, and test datasets prepared during the data preparation stage. The original dataset will not be re-split.

The workflow is:

**Train → TF-IDF → Classifier Training → Validation & Model Comparison → Model Selection → Final Test Evaluation**

The primary model-selection metric is **Macro F1**, complemented by accuracy and per-class performance metrics.


## Setup and Data Loading

This section imports the libraries required for modelling and evaluation, locates the prepared dataset files, and loads the predefined train, validation, and test splits.

In [2]:
from pathlib import Path

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import matplotlib.pyplot as plt


In [3]:
Path.cwd()

PosixPath('/Users/cristianalavrador/Desktop/ExecutiveMaster PBS/Courses/Term 3/Machine Learning/Group work/bankassist-triage/notebooks')

### Load Predefined Data Splits

The modelling stage uses the train, validation, and test splits produced during the data preparation stage. These splits are kept unchanged to ensure consistency across the project and to avoid data leakage.

- **Training set:** used to fit the models.
- **Validation set:** used to compare model configurations and select the best-performing approach.
- **Test set:** reserved for the final unbiased evaluation of the selected model.

In [4]:
DATA_DIR = Path("../data/splits")

train_path = DATA_DIR / "train.csv"
validation_path = DATA_DIR / "validation.csv"
test_path = DATA_DIR / "test.csv"

print("Train:", train_path.exists())
print("Validation:", validation_path.exists())
print("Test:", test_path.exists())

Train: True
Validation: True
Test: True


In [5]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Train shape: (696, 2)
Validation shape: (149, 2)
Test shape: (150, 2)


In [6]:
print("Columns:", train_df.columns.tolist())

display(train_df.head())

Columns: ['text', 'label']


,text,label
0,how can i transfer money to a beneficiary?,beneficiary_not_allowed
1,"apple watch top up help, please?",apple_pay_or_google_pay
2,can i order a second card?,getting_spare_card
3,what card or currency can i use to pay?,supported_cards_and_currencies
4,how was i charged extra when i used my card to...,card_payment_fee_charged


### Features and Target

The input feature (`X`) is the customer message stored in the `text` column, while the target variable (`y`) is the corresponding banking intent stored in the `label` column.

The same structure is maintained across the training, validation, and test sets.

In [7]:
X_train = train_df["text"]
y_train = train_df["label"]

X_validation = validation_df["text"]
y_validation = validation_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))
print("Test samples:", len(X_test))
print("Number of classes:", y_train.nunique())

Training samples: 696
Validation samples: 149
Test samples: 150
Number of classes: 10


In [8]:
classes = sorted(y_train.unique())

print("Classes:")
for label in classes:
    print("-", label)

Classes:
- apple_pay_or_google_pay
- beneficiary_not_allowed
- cancel_transfer
- card_about_to_expire
- card_payment_fee_charged
- change_pin
- getting_spare_card
- lost_or_stolen_phone
- supported_cards_and_currencies
- visa_or_mastercard


## Model Development

The modelling approach combines **TF-IDF (Term Frequency–Inverse Document Frequency)** for text feature extraction with **Logistic Regression** for multiclass intent classification.

TF-IDF transforms each customer message into a numerical feature vector based on the importance of words within the dataset. Logistic Regression then uses these features to estimate the probability of each of the 10 intent classes.

Two model configurations will be trained on the same training set and compared using the validation set:

- **Model A — Baseline:** TF-IDF using unigrams (single words) + Logistic Regression.
- **Model B — Extended:** TF-IDF using unigrams and bigrams (single words and two-word combinations) + Logistic Regression.

The models will be compared primarily using **Macro F1**. The test set will remain untouched during model selection and will only be used for the final evaluation of the selected model.